# Project 59 — Local CRM Enrichment Agent

**Summarize account info, analyze engagement, and suggest next actions — locally.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Data | In-memory CRM records |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to build a **CRM enrichment pipeline** that summarizes customer accounts
2. How to **analyze interaction history** and identify engagement patterns
3. How to generate **next-best-action recommendations** using an LLM
4. How to **score accounts** for churn risk and upsell opportunity
5. How to produce **account briefs** for sales teams

## 2 · Why This Project Matters

Sales teams spend hours reading through CRM notes, emails, and interaction logs
before each call. An enrichment agent can automatically summarize this data and
suggest the best next action — saving time and improving close rates. Running
locally keeps sensitive customer data secure.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`

In [ ]:
# !pip install -q langchain langchain-ollama

## 4 · Imports and Configuration

In [ ]:
import json
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.2)
print(f'LLM ready: {MODEL}')

## 5 · Sample CRM Data

We create realistic CRM records with company info, interactions, deals, and notes.

In [ ]:
CRM_ACCOUNTS = [
    {
        'company': 'TechStart Inc.',
        'industry': 'SaaS',
        'size': '50-100 employees',
        'annual_revenue': '$5M',
        'current_plan': 'Professional ($500/mo)',
        'contract_end': '2024-08-15',
        'health_score': 72,
        'nps_score': 7,
        'primary_contact': 'Sarah Chen, VP Engineering',
        'interactions': [
            {'date': '2024-03-01', 'type': 'call', 'summary': 'Quarterly review. Happy with product. Asked about API rate limits.'},
            {'date': '2024-02-15', 'type': 'support', 'summary': 'Reported slow dashboard loading. Resolved in 2 hours.'},
            {'date': '2024-01-20', 'type': 'email', 'summary': 'Inquired about enterprise plan pricing and SSO features.'},
            {'date': '2024-01-05', 'type': 'call', 'summary': 'Onboarded 10 new team members. Requested training session.'},
        ],
        'deals': [
            {'name': 'Enterprise Upgrade', 'stage': 'Proposal', 'value': '$1,200/mo', 'probability': 60},
        ],
    },
    {
        'company': 'GlobalRetail Corp',
        'industry': 'Retail',
        'size': '500-1000 employees',
        'annual_revenue': '$50M',
        'current_plan': 'Enterprise ($2,000/mo)',
        'contract_end': '2024-06-30',
        'health_score': 45,
        'nps_score': 4,
        'primary_contact': 'Mike Johnson, CTO',
        'interactions': [
            {'date': '2024-03-10', 'type': 'support', 'summary': 'Third ticket this month about data sync issues. Escalated to engineering.'},
            {'date': '2024-03-05', 'type': 'call', 'summary': 'Frustration expressed about recurring bugs. Mentioned evaluating competitors.'},
            {'date': '2024-02-20', 'type': 'support', 'summary': 'Critical bug in reporting module. Customer lost 2 hours of work.'},
            {'date': '2024-02-01', 'type': 'email', 'summary': 'Contract renewal discussion. Asked for 20% discount or will consider alternatives.'},
        ],
        'deals': [
            {'name': 'Contract Renewal', 'stage': 'At Risk', 'value': '$24,000/yr', 'probability': 35},
        ],
    },
    {
        'company': 'HealthFirst Labs',
        'industry': 'Healthcare',
        'size': '100-250 employees',
        'annual_revenue': '$15M',
        'current_plan': 'Starter ($200/mo)',
        'contract_end': '2025-01-15',
        'health_score': 88,
        'nps_score': 9,
        'primary_contact': 'Dr. Lisa Park, Head of Data',
        'interactions': [
            {'date': '2024-03-12', 'type': 'call', 'summary': 'Extremely happy. Using product daily. Wants to add 3 more departments.'},
            {'date': '2024-02-28', 'type': 'email', 'summary': 'Requested case study participation. Great advocate.'},
            {'date': '2024-02-10', 'type': 'call', 'summary': 'Asked about HIPAA compliance features for clinical data.'},
        ],
        'deals': [
            {'name': 'Multi-dept Expansion', 'stage': 'Discovery', 'value': '$800/mo', 'probability': 75},
        ],
    },
]

print(f'Loaded {len(CRM_ACCOUNTS)} CRM accounts:')
for acct in CRM_ACCOUNTS:
    print(f'  {acct["company"]} | Health: {acct["health_score"]} | Plan: {acct["current_plan"]}')

## 6 · Build the Enrichment Pipeline

For each account, the agent:
1. Summarizes the account context
2. Analyzes engagement patterns
3. Assesses churn risk and upsell potential
4. Recommends next actions

In [ ]:
def enrich_account(account: dict) -> dict:
    """Run the full enrichment pipeline on a CRM account."""
    acct_json = json.dumps(account, indent=2)

    # Step 1: Account Summary
    summary_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a CRM analyst. Produce a concise 3-sentence account summary '
         'highlighting the key facts a sales rep needs before a call.'),
        ('human', '{account}'),
    ])
    summary = (summary_prompt | llm | StrOutputParser()).invoke({'account': acct_json})

    # Step 2: Engagement Analysis
    engagement_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a customer success analyst. Analyze the interaction history and identify:\n'
         '1. Engagement trend (increasing/stable/declining)\n'
         '2. Key themes in interactions\n'
         '3. Red flags or positive signals\n'
         'Be concise and data-driven.'),
        ('human', '{account}'),
    ])
    engagement = (engagement_prompt | llm | StrOutputParser()).invoke({'account': acct_json})

    # Step 3: Risk & Opportunity Scoring
    score_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a revenue analyst. Given this account, respond with ONLY a JSON '
         'object (no markdown):\n'
         '{{"churn_risk": "low|medium|high", "churn_reason": "...", '
         '"upsell_potential": "low|medium|high", "upsell_reason": "...", '
         '"priority": "low|medium|high|critical"}}'),
        ('human', '{account}'),
    ])
    score_raw = (score_prompt | llm | StrOutputParser()).invoke({'account': acct_json})
    try:
        start = score_raw.find('{')
        end = score_raw.rfind('}') + 1
        scores = json.loads(score_raw[start:end]) if start >= 0 else {}
    except (json.JSONDecodeError, ValueError):
        scores = {'raw': score_raw[:200]}

    # Step 4: Next Actions
    action_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a sales coach. Given this account analysis, recommend the top 3 '
         'next actions the account manager should take. Be specific and actionable.'),
        ('human',
         'Account: {company}\n'
         'Summary: {summary}\n'
         'Engagement: {engagement}\n'
         'Scores: {scores}'),
    ])
    actions = (action_prompt | llm | StrOutputParser()).invoke({
        'company': account['company'],
        'summary': summary,
        'engagement': engagement,
        'scores': json.dumps(scores),
    })

    return {
        'company': account['company'],
        'summary': summary,
        'engagement_analysis': engagement,
        'scores': scores,
        'next_actions': actions,
    }


print('Enrichment pipeline ready.')

## 7 · Run Enrichment on All Accounts

In [ ]:
enriched = []
for acct in CRM_ACCOUNTS:
    print(f'\nEnriching: {acct["company"]}...')
    result = enrich_account(acct)
    enriched.append(result)

    print(f'\n{"="*60}')
    print(f'ACCOUNT BRIEF: {result["company"]}')
    print(f'{"="*60}')
    print(f'\nSummary: {result["summary"]}')
    print(f'\nEngagement: {result["engagement_analysis"]}')
    print(f'\nScores: {json.dumps(result["scores"], indent=2)}')
    print(f'\nNext Actions: {result["next_actions"]}')

## 8 · Portfolio Overview

Generate a portfolio-level summary across all accounts.

In [ ]:
portfolio_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a VP of Customer Success. Given enrichment data for multiple accounts, '
     'produce a portfolio review that includes:\n'
     '1. Overall portfolio health\n'
     '2. Accounts needing immediate attention\n'
     '3. Top expansion opportunities\n'
     '4. This week\'s priority actions\n'
     'Be specific with account names and numbers.'),
    ('human', '{portfolio}'),
])
chain = portfolio_prompt | llm | StrOutputParser()
portfolio_review = chain.invoke({'portfolio': json.dumps(enriched, indent=2)})

print('PORTFOLIO REVIEW')
print('=' * 50)
print(portfolio_review)

## 9 · Save Results

In [ ]:
output = {
    'enriched_accounts': enriched,
    'portfolio_review': portfolio_review,
}
with open('crm_enrichment_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print('Results saved.')

## 10 · Limitations

- **Mocked CRM data** — not connected to Salesforce/HubSpot
- **No email integration** — doesn't analyze actual email threads
- **Scoring is LLM-based** — not calibrated with historical outcomes
- **Small context** — real accounts may have hundreds of interactions
- No A/B testing of recommendations

## 11 · How to Improve

1. **Connect to real CRM** via Salesforce/HubSpot API
2. **Add email analysis** by parsing email threads
3. **Train churn model** on historical data for calibrated scoring
4. **Integrate with calendar** to auto-schedule recommended follow-ups
5. **Use RAG** over interaction history for very large accounts

## 12 · Mini Challenge

1. Add a 'competitor mention tracker' that flags competitor names in interactions
2. Generate a personalized email draft for the top-priority account
3. Add a time-decay factor to engagement analysis (recent interactions matter more)

## 13 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| CRM enrichment | Multi-step analysis of customer accounts |
| Structured scoring | LLM produces JSON risk/opportunity scores |
| Action generation | AI suggests specific next-best-actions |
| Portfolio analysis | Aggregate view across multiple accounts |
| Local-first | Sensitive CRM data stays on your machine |